# Day 5 — Linear Algebra for Machine Learning
**30-Day ML & AI Mastery Plan · Shubham**

To a computer, **all data is numbers in grids**: a customer is a row vector, a dataset is a matrix, an image is a matrix of pixels, a sentence becomes a TF-IDF or embedding vector. Every model is then a chain of matrix operations: a linear model is `X @ w + b`, and a neural network is many of those with a non-linearity in between. Today you learn that language, and you'll **see** every operation as geometry.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | Vectors: data rows as arrows | 15 min |
| 2 | Norms (L1, L2, L∞) and distances | 20 min |
| 3 | Dot product, angles and cosine similarity | 25 min |
| 4 | Matrices and matrix multiplication | 25 min |
| 5 | Matrices as transformations; the determinant | 25 min |
| **6** | **Live demo: rotate, scale and shear an image with matrices** | 30 min |
| 7 | Inverse and solving linear systems | 20 min |
| 8 | Rank and linear (in)dependence: multicollinearity | 20 min |
| 9 | Eigenvalues, eigenvectors and PCA | 35 min |
| 10 | Exercises | 30–60 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(seed=5)

def draw_arrows(vectors, colors, labels, ax, lim=4):
    """Draw 2-D vectors as arrows from the origin."""
    for v, c, l in zip(vectors, colors, labels):
        ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=c, label=l, width=0.012)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8); ax.legend(loc="lower left")

---
# Part 1 — Vectors

A **vector** is an ordered list of numbers. Two ways to see it:
- **As data:** one customer = `[age, income, visits]`. One email = 200,000 TF-IDF numbers.
- **As geometry:** an arrow from the origin. `[3, 1]` means "3 right, 1 up".

Both views are the same thing, and switching between them is the core skill of today.

| Operation | Maths | NumPy | Geometry |
|---|---|---|---|
| add | u + v | `u + v` | put the arrows head to tail |
| scale | c·v | `c * v` | stretch (c > 1), shrink, or flip (c < 0) |
| linear combination | a·u + b·v | `a*u + b*v` | any point reachable by mixing u and v |

In [ ]:
u = np.array([3, 1])
v = np.array([1, 2])
print("u + v =", u + v)
print("2 * u =", 2 * u)
print("-1 * v =", -1 * v)
print("0.5u + 1.5v =", 0.5 * u + 1.5 * v)

fig, ax = plt.subplots(figsize=(5, 5))
draw_arrows([u, v, u + v, -v], ["tab:blue", "tab:orange", "tab:green", "tab:red"], ["u", "v", "u + v", "-v"], ax)
ax.plot([u[0], u[0] + v[0]], [u[1], u[1] + v[1]], "k--", lw=0.8)       # v placed at the tip of u
ax.set_title("Adding vectors = head to tail"); plt.show()

In [ ]:
# A dataset is a stack of vectors. Each ROW is one sample, each COLUMN one feature.
students = np.array([[72, 85, 90],     # Asha:  math, physics, english
                     [55, 60, 70],     # Ravi
                     [90, 95, 80]])    # Meena
print("one student (row vector):", students[0])
print("one feature (column vector):", students[:, 0])
print("shape (samples, features):", students.shape)

---
# Part 2 — Norms: how long is a vector?

A **norm** measures the size of a vector. The distance between two points is the norm of their difference.

| Norm | Formula | Name | ML use |
|---|---|---|---|
| **L2** | √(x₁² + x₂² + …) | Euclidean, "as the crow flies" | k-NN distance, Ridge regularisation |
| **L1** | \|x₁\| + \|x₂\| + … | Manhattan, "city blocks" | Lasso regularisation (pushes weights to exactly 0) |
| **L∞** | max \|xᵢ\| | Chebyshev, "the worst coordinate" | the maximum error; adversarial robustness |

In [ ]:
x = np.array([3, -4])
print("L2 :", np.linalg.norm(x), "= sqrt(9 + 16)")
print("L1 :", np.linalg.norm(x, 1), "= |3| + |-4|")
print("Linf:", np.linalg.norm(x, np.inf), "= max(|3|, |-4|)")

a, b = np.array([1, 1]), np.array([4, 5])
print("\nEuclidean distance a->b:", np.linalg.norm(a - b))
print("Manhattan distance a->b:", np.linalg.norm(a - b, 1))

unit = x / np.linalg.norm(x)          # same direction, length 1
print("\nunit vector:", unit, "length", np.linalg.norm(unit))

### What a "unit ball" looks like in each norm
All the points with norm exactly 1. This picture explains Lasso: the L1 ball has **corners on the axes**, so when a model is pushed to keep its weights inside it, the best point often sits on a corner, where some weights are exactly 0. That's automatic feature selection.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.cos(theta), np.sin(theta), label="L2: circle")
ax.plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], label="L1: diamond (corners on the axes)")
ax.plot([1, 1, -1, -1, 1], [1, -1, -1, 1, 1], label="L∞: square")
ax.set_aspect("equal"); ax.legend(loc="upper right", fontsize=8); ax.set_title("Points with norm = 1"); plt.show()

---
# Part 3 — The dot product: the most important operation in ML

$$u \cdot v = u_1 v_1 + u_2 v_2 + \dots + u_n v_n = \lVert u \rVert\,\lVert v \rVert \cos\theta$$

Two readings of the same number:
- **Algebra:** multiply matching entries and add. That's exactly a linear model: `score = features · weights`.
- **Geometry:** how much the two vectors point the same way. Positive = similar direction, 0 = perpendicular (orthogonal), negative = opposite.

**Cosine similarity** removes the lengths and keeps only the angle: cos θ = (u · v) / (‖u‖‖v‖), from −1 to 1.

In [ ]:
u, v = np.array([3, 1]), np.array([1, 2])
print("u . v by hand :", 3*1 + 1*2)
print("u . v in NumPy:", u @ v, "| np.dot:", np.dot(u, v))
cos = u @ v / (np.linalg.norm(u) * np.linalg.norm(v))
print(f"cos(theta) = {cos:.3f} -> angle = {np.degrees(np.arccos(cos)):.1f} degrees")
print("perpendicular: [1, 0] . [0, 1] =", np.array([1, 0]) @ np.array([0, 1]))

In [ ]:
# A linear model IS a dot product: score = x . w + b  (this is TextPulse's logistic regression)
features = np.array([0.6, 0.0, 0.8])          # TF-IDF values for "free", "meeting", "prize"
weights = np.array([1.2, -1.5, 0.9])          # learned weights
bias = -0.4
score = features @ weights + bias
print("score z =", score, "-> P(spam) =", round(1 / (1 + np.exp(-score)), 3))

### Live mini-demo: which sentences mean similar things?
Search engines, recommendation systems and RAG chatbots (Day 28) all work like this: turn text into vectors, then find the vectors with the **highest cosine similarity**. Here with TF-IDF vectors, the same idea TextPulse uses:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
docs = ["win a free prize now",
        "claim your free prize today",
        "team meeting moved to friday",
        "friday team lunch is cancelled",
        "the cricket match is on friday"]
tfidf = TfidfVectorizer().fit_transform(docs).toarray()     # each row: one unit-length vector
similarity = tfidf @ tfidf.T                                  # all pairwise dot products at once
pd.DataFrame(similarity, index=[d[:22] for d in docs], columns=range(5)).round(2)

Because TF-IDF rows are already unit length (L2-normalised), the dot product **is** the cosine similarity. "win a free prize now" is most similar to the other prize message; the two team messages pair up. One matrix multiplication computed all 25 comparisons.

**Projection** (how much of v lies along u) also comes from the dot product: proj = (v · û) û. Linear regression is a projection of the target onto the space of the features (Day 8).

---
# Part 4 — Matrices and matrix multiplication

A matrix is a grid of numbers with shape **(rows, columns)**. A dataset `X` with n samples and d features is an (n, d) matrix.

**Matrix multiplication** `A @ B`: entry (i, j) is the **dot product of row i of A with column j of B**.
- Shapes must match on the inside: **(n, d) @ (d, k) → (n, k)**.
- It's **not commutative**: A @ B ≠ B @ A in general.
- `A * B` is something else: element-wise multiplication (same shapes, or broadcasting).

The **transpose** `A.T` swaps rows and columns: (n, d) → (d, n). Useful identity: (AB)ᵀ = BᵀAᵀ.

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])          # (3, 2)
B = np.array([[1, 0, 2],
              [0, 1, 3]])       # (2, 3)
print("A @ B  (3,2)@(2,3) -> (3,3):\n", A @ B)
print("B @ A  (2,3)@(3,2) -> (2,2):\n", B @ A)
print("entry (0,2) of A@B by hand: row 0 of A . column 2 of B =", A[0] @ B[:, 2])
print("A * A (element-wise):\n", A * A)
try:
    A @ A
except ValueError as e:
    print("A @ A fails:", str(e)[:70], "...")

In [ ]:
# Why ML loves matrix multiplication: predictions for EVERY row in one operation
X = rng.normal(size=(5, 3))          # 5 samples, 3 features
w = np.array([0.5, -1.0, 2.0])       # one weight per feature
b = 0.1
print("one row at a time:", [round(float(row @ w + b), 3) for row in X])
print("all at once      :", X @ w + b)

# A layer of a neural network: 3 inputs -> 4 neurons is a (3, 4) weight matrix
W = rng.normal(size=(3, 4))
print("layer output shape:", (X @ W).shape, "= (samples, neurons)")

In [ ]:
I = np.eye(3)
print("identity matrix:\n", I)
print("X @ I == X:", np.allclose(X @ I, X))
print("(A @ B).T == B.T @ A.T:", np.allclose((A @ B).T, B.T @ A.T))

---
# Part 5 — Matrices as transformations

A 2×2 matrix **moves every point of the plane**. The trick to reading any matrix: **its columns show where the two basis vectors land.** Column 1 is where `[1, 0]` goes; column 2 is where `[0, 1]` goes. Everything else follows along, because the transformation is **linear**: straight lines stay straight, the origin stays fixed, and grid lines stay parallel and evenly spaced.

| Transformation | Matrix | Effect |
|---|---|---|
| Scale | [[sx, 0], [0, sy]] | stretch along x and y |
| Rotate by θ | [[cos θ, −sin θ], [sin θ, cos θ]] | turn anticlockwise |
| Shear | [[1, k], [0, 1]] | slide rows sideways in proportion to height |
| Reflect | [[1, 0], [0, −1]] | mirror across the x-axis |
| Project | [[1, 0], [0, 0]] | flatten onto the x-axis (loses information) |

The **determinant** det(A) tells you how much areas are multiplied. det = 2 doubles areas, det < 0 flips orientation (a mirror), **det = 0 squashes the plane onto a line**: information is lost and the matrix has no inverse.

In [ ]:
house = np.array([[0, 0], [2, 0], [2, 2], [1, 3], [0, 2], [0, 0]]).T     # 2 x 6: one column per corner

theta = np.radians(30)
transforms = {
    "scale (1.5, 0.5)": np.array([[1.5, 0], [0, 0.5]]),
    "rotate 30°": np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]),
    "shear k = 0.8": np.array([[1, 0.8], [0, 1]]),
    "reflect in x-axis": np.array([[1, 0], [0, -1]]),
    "project onto x": np.array([[1, 0], [0, 0]]),
}
fig, axes = plt.subplots(1, 5, figsize=(19, 4))
for ax, (name, M) in zip(axes, transforms.items()):
    moved = M @ house                                 # transform every corner at once
    ax.fill(house[0], house[1], alpha=0.25, label="original")
    ax.fill(moved[0], moved[1], alpha=0.5, label="transformed")
    ax.plot(moved[0], moved[1], color="tab:orange", lw=3)    # outline: still visible when flattened to a line
    ax.set_xlim(-2, 4.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
    ax.set_title(f"{name}\ndet = {np.linalg.det(M):.2f}")
axes[0].legend(loc="lower right"); plt.tight_layout(); plt.show()

Check the determinants against the pictures: scaling by 1.5 × 0.5 gives area × 0.75; rotation and shear keep the area (det = 1); the reflection has det = −1 (same area, flipped); the projection has det = 0 (the house is flattened to a line).

### Composition: one matrix per step, multiply them
Doing transform A, **then** B, is the single matrix **B @ A** (the rightmost acts first). Order matters:

In [ ]:
R = transforms["rotate 30°"]; S = transforms["shear k = 0.8"]
print("rotate then shear (S @ R):\n", S @ R)
print("shear then rotate (R @ S):\n", R @ S)
print("same? ", np.allclose(S @ R, R @ S))
print("det(S @ R) = det(S) * det(R):", round(np.linalg.det(S @ R), 3), "=", round(np.linalg.det(S) * np.linalg.det(R), 3))

---
# Part 6 — LIVE DEMO: rotate, scale and shear an image with matrices

An image is a grid of pixels. Each pixel has a position (row, column) and a colour. To transform the image with a matrix M, we move the **positions**.

**The catch:** if you push every source pixel forward through M, the results land between grid points and leave holes. Real image libraries go **backwards**: for every pixel of the output, ask "where did this come from?" using **M⁻¹** (the inverse), and copy that colour. So image transformation is a direct use of the matrix inverse.

In [ ]:
from sklearn.datasets import load_sample_image
img = load_sample_image("china.jpg")[::2, ::2]           # halve the resolution to keep it fast
print("image shape (height, width, colours):", img.shape)
plt.figure(figsize=(6, 4)); plt.imshow(img); plt.title("Original"); plt.axis("off"); plt.show()

In [ ]:
def transform_image(img, M):
    """Apply a 2x2 matrix M to an image, around the image centre, using inverse mapping."""
    h, w = img.shape[:2]
    centre = np.array([w / 2, h / 2])
    # Coordinates of every OUTPUT pixel, as a (2, h*w) matrix of (x, y) columns
    ys, xs = np.mgrid[0:h, 0:w]
    out_coords = np.vstack([xs.ravel(), ys.ravel()]) - centre[:, None]
    # Where did each output pixel come from?  source = M^-1 @ output
    src = np.linalg.inv(M) @ out_coords + centre[:, None]
    sx, sy = np.round(src).astype(int)                     # nearest source pixel
    inside = (sx >= 0) & (sx < w) & (sy >= 0) & (sy < h)
    out = np.full_like(img, 255)                            # white background
    out.reshape(-1, img.shape[2])[inside] = img[sy[inside], sx[inside]]
    return out

In [ ]:
theta = np.radians(25)
# Image rows grow DOWNWARD, so a positive angle here turns the picture clockwise on screen
demos = {
    "Rotate 25°": np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]),
    "Scale x0.6, y1.0": np.array([[0.6, 0], [0, 1.0]]),
    "Shear k = 0.4": np.array([[1, 0.4], [0, 1]]),
    "Mirror (flip x)": np.array([[-1, 0], [0, 1]]),
    "Rotate, then shear": np.array([[1, 0.4], [0, 1]]) @ np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]),
}
fig, axes = plt.subplots(2, 3, figsize=(16, 7.5))
axes[0, 0].imshow(img); axes[0, 0].set_title("Original")
for ax, (name, M) in zip(axes.ravel()[1:], demos.items()):
    ax.imshow(transform_image(img, M))
    ax.set_title(f"{name}   det = {np.linalg.det(M):.2f}")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Same result with SciPy's optimised version (it also uses the inverse matrix, plus smooth interpolation)
from scipy import ndimage
M = demos["Rotate 25°"]
h, w = img.shape[:2]
centre = np.array([h / 2, w / 2])
Minv_rc = np.linalg.inv(M)[::-1, ::-1]                    # SciPy works in (row, col) order, ours was (x, y)
offset = centre - Minv_rc @ centre
scipy_version = np.stack([ndimage.affine_transform(img[..., c], Minv_rc, offset=offset, order=1, cval=255)
                          for c in range(3)], axis=-1)
ours = transform_image(img, M)
agree = (np.abs(ours.astype(int) - scipy_version.astype(int)).mean(axis=-1) < 40).mean()
print(f"pixels where our version and SciPy's agree closely: {agree:.1%}")

Every photo app, the data augmentation used to train image models (Day 18: random rotations, flips and shears so a model learns that a rotated cat is still a cat), and computer graphics all do exactly this. The determinant printed above each picture tells you how the area changed: the mirror has det = −1 (flipped), the squeeze has 0.6.

---
# Part 7 — Inverse and solving linear systems

The **inverse** A⁻¹ undoes A: A⁻¹A = I. It exists only for **square** matrices with **det ≠ 0** (nothing was squashed flat, so the transformation can be reversed).

A system of linear equations is one matrix equation **Ax = b**:
```
2x +  y = 5          [[2, 1],   [x,   [5,
 x + 3y = 10    ->    [1, 3]] @  y] =  10]
```

In [ ]:
A = np.array([[2, 1], [1, 3]])
b = np.array([5, 10])
x = np.linalg.solve(A, b)                 # preferred: faster and more accurate than inv(A) @ b
print("solution:", x, "| check A @ x =", A @ x)
print("A^-1:\n", np.linalg.inv(A))
print("A^-1 @ A = I:", np.allclose(np.linalg.inv(A) @ A, np.eye(2)))

In [ ]:
# Singular (not invertible): the second row is 2x the first, so both equations say the same thing
S = np.array([[1, 2], [2, 4]])
print("det =", np.linalg.det(S))
try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("inverse fails:", e)

# Nearly singular: the answer becomes hugely sensitive to tiny changes in the data
N = np.array([[1, 1], [1, 1.0001]])
print("\ncondition number:", round(np.linalg.cond(N)))
print("solve with b = [2, 2.0001]:", np.linalg.solve(N, [2, 2.0001]))
print("solve with b = [2, 2.0002]:", np.linalg.solve(N, [2, 2.0002]), "<- tiny change in b, big change in x")

**Why it matters:** linear regression has a closed-form solution, w = (XᵀX)⁻¹Xᵀy (the "normal equation", Day 8). If two features are (almost) copies of each other, XᵀX is (nearly) singular: the weights become huge, unstable and meaningless. That's **multicollinearity**, and rank tells you when it happens.

---
# Part 8 — Rank and linear (in)dependence

Vectors are **linearly dependent** if one of them can be built from the others (e.g. c = a + b). Then it adds **no new information**.

The **rank** of a matrix = the number of linearly independent columns (= independent rows) = the number of genuinely different directions it contains.
- (n, d) data with rank d: every feature carries new information. **Full rank.**
- rank < d: some feature is a combination of the others. **Rank-deficient**: XᵀX can't be inverted.

In [ ]:
a = np.array([1, 0, 2]); b = np.array([0, 1, 1]); c = a + b
print("rank of [a, b]    :", np.linalg.matrix_rank(np.column_stack([a, b])))
print("rank of [a, b, c] :", np.linalg.matrix_rank(np.column_stack([a, b, c])), "<- c = a + b adds nothing new")
print("rank of identity 3:", np.linalg.matrix_rank(np.eye(3)))

In [ ]:
# A real case: Titanic. family_size = sibsp + parch + 1 is an EXACT combination of other columns
titanic = sns.load_dataset("titanic")
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
X_fam = np.column_stack([np.ones(len(titanic)), titanic[["sibsp", "parch", "family_size"]].to_numpy()])
print("columns: [intercept, sibsp, parch, family_size] -> shape", X_fam.shape)
print("rank:", np.linalg.matrix_rank(X_fam), "of 4  <- one column is redundant")
print("condition number of X^T X:", f"{np.linalg.cond(X_fam.T @ X_fam):.1e}", "(huge = effectively singular)")

Because `family_size = 1·intercept + sibsp + parch`, the four columns only span 3 directions. A linear model can't tell how to split credit between them: infinitely many weight combinations give identical predictions. Fixes: **drop one column**, or use **regularisation** (Ridge adds λI to XᵀX, which makes it invertible again).

The same trap appears with one-hot encoding: if you keep a dummy column for **every** category plus an intercept, the dummies always add up to 1 = the intercept. That's the "dummy variable trap", and why `pd.get_dummies(..., drop_first=True)` exists.

---
# Part 9 — Eigenvalues and eigenvectors, and PCA

Most vectors change direction when a matrix acts on them. **Eigenvectors** are the special directions that only get **stretched or shrunk**, not turned:

$$A v = \lambda v$$

v is the eigenvector, λ (lambda) the eigenvalue: the stretch factor along that direction. λ = 2 doubles it, λ = 0.5 halves it, λ < 0 flips it.

In [ ]:
A = np.array([[3, 1],
              [0, 2]])
eigenvalues, eigenvectors = np.linalg.eig(A)
# Some NumPy versions return complex numbers with a zero imaginary part; keep the real part
eigenvalues, eigenvectors = np.real_if_close(eigenvalues), np.real_if_close(eigenvectors)
print("eigenvalues:", eigenvalues)
print("eigenvectors (columns):\n", eigenvectors)
for i in range(2):
    v = eigenvectors[:, i]
    print(f"A @ v{i} = {A @ v}   lambda * v{i} = {eigenvalues[i] * v}")

# Picture: an ordinary vector gets turned; eigenvectors keep their direction
fig, ax = plt.subplots(figsize=(5, 5))
other = np.array([-1, 1.5])
draw_arrows([other, A @ other, eigenvectors[:, 0], A @ eigenvectors[:, 0], eigenvectors[:, 1], A @ eigenvectors[:, 1]],
            ["grey", "black", "tab:blue", "tab:cyan", "tab:red", "salmon"],
            ["x", "A x (turned)", "v0", "A v0 = 3 v0", "v1", "A v1 = 2 v1"], ax, lim=4)
plt.show()

### Why ML cares: PCA
For a dataset's **covariance matrix** (Day 4's spread, for every pair of features), the eigenvectors point along the directions where the data **varies most**, and each eigenvalue is the variance along its direction. Covariance matrices are symmetric, so their eigenvectors are perpendicular and the eigenvalues are real and ≥ 0 (use `np.linalg.eigh` for symmetric matrices).

**Principal Component Analysis (PCA, Day 13)** = keep the top few eigenvectors and project the data onto them. 30 features → 2, while keeping most of the information.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
data = load_breast_cancer()
Xs = StandardScaler().fit_transform(data.data)            # standardise first (Day 4), else 'area' dominates

cov = np.cov(Xs, rowvar=False)                             # 30 x 30 covariance matrix
vals, vecs = np.linalg.eigh(cov)                           # eigh: for symmetric matrices
order = np.argsort(vals)[::-1]                             # biggest variance first
vals, vecs = vals[order], vecs[:, order]

explained = vals / vals.sum()
print("variance explained by the first 5 directions:", explained[:5].round(3))
print("first 2 together:", explained[:2].sum().round(3), "| first 10:", explained[:10].sum().round(3))

In [ ]:
projected = Xs @ vecs[:, :2]                               # PCA is just a matrix multiplication
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].bar(range(1, 11), explained[:10]); axes[0].plot(range(1, 11), np.cumsum(explained[:10]), "o-", color="red", label="cumulative")
axes[0].set(title="Variance explained by each eigenvector", xlabel="component", ylabel="share of variance"); axes[0].legend()
for label, name in enumerate(data.target_names):
    pts = projected[data.target == label]
    axes[1].scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.6, label=name)
axes[1].set(title="569 tumours, 30 features squeezed to 2", xlabel="PC 1", ylabel="PC 2"); axes[1].legend()
plt.tight_layout(); plt.show()

from sklearn.decomposition import PCA
sk = PCA(n_components=2).fit(Xs)
print("sklearn PCA explained variance:", sk.explained_variance_ratio_.round(3), "<- same as our eigenvalues")
print("same directions (up to sign):", np.allclose(np.abs(sk.components_), np.abs(vecs[:, :2].T)))

Two directions out of 30 keep about **63%** of all the variation, and the scatter plot already separates malignant from benign tumours almost cleanly, with no labels used to find those directions. That's the power of eigenvectors: they find the structure in the data on their own.

### Where today's ideas appear later
| Concept | Shows up in |
|---|---|
| dot product | every linear model, attention in transformers (Day 24), similarity search in RAG (Day 28) |
| norms | k-NN and k-means distances (Days 8, 12), L1/L2 regularisation (Day 9) |
| matrix multiply | every neural-network layer (Days 16–20) |
| inverse / solve | the normal equation for linear regression (Day 8) |
| rank | multicollinearity, the dummy-variable trap, low-rank fine-tuning of LLMs (LoRA, Day 26) |
| eigenvectors | PCA (Day 13), Google's PageRank, spectral clustering |

---
# Part 10 — Exercises
Paste your code in chat for review.

**Ex 1 — Norms and similarity.** For u = [2, −1, 3] and v = [1, 4, −2]: compute the L1, L2 and L∞ norms of each, the dot product, and the angle between them in degrees. Are they closer to parallel or perpendicular?

**Ex 2 — Transformations.** Build ONE matrix that scales by 2 along x, then rotates by 90°. Apply it to the house shape and plot before and after. Predict its determinant before computing it.

**Ex 3 — Image.** Use `transform_image` to make the photo look like it's being viewed in a "funhouse mirror": squeeze it to 50% width and shear it by 0.3. Then apply the **inverse** matrix to the result. Do you get the original back exactly? Why not?

**Ex 4 — Rank.** On Titanic, build a matrix with an intercept column plus one-hot columns for **all three** values of `embark_town` (drop missing rows). Compute its rank. Then use `drop_first=True` and compute it again. Explain the difference in one sentence.

**Stretch — PCA by hand.** Repeat Part 9 on `sklearn.datasets.load_wine()`. How many components do you need to keep 80% of the variance? Plot the first two components coloured by wine class.

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Stretch

---
### Day 5 recap
- **Vector** = a data row = an arrow. A dataset is a matrix of shape (samples, features).
- **Norms:** L2 (straight line), L1 (city blocks; Lasso → zero weights), L∞ (largest coordinate). Distance = norm of the difference.
- **Dot product** = Σ uᵢvᵢ = ‖u‖‖v‖cos θ. A linear model is a dot product; cosine similarity powers search and recommendations.
- **Matrix multiply:** (n, d) @ (d, k) → (n, k); row-by-column dot products; order matters; `*` is element-wise.
- **A matrix is a transformation:** its columns are where the basis vectors land. det = area scale; det < 0 flips; det = 0 squashes and can't be undone.
- **Inverse** undoes a transformation; solve Ax = b with `np.linalg.solve`. Image transforms use the inverse to map output pixels back to the source.
- **Rank** = number of independent columns. Redundant features (family_size, all dummies + intercept) make XᵀX singular: drop a column or regularise.
- **Eigenvectors** keep their direction (Av = λv). Of a covariance matrix → the directions of greatest variance → **PCA**.
- **Tomorrow (Day 6): Calculus.** Derivatives, the chain rule and gradients, ending with gradient descent written from scratch.